In [14]:
import pandas as pd
import os
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\jhigh\Projects\triathlon-db\Mixed_Relay


In [30]:
# File path
file_path = "USAT_MixedRelay_vJH.xlsx"

# Read all sheet names
all_sheets = pd.ExcelFile(file_path).sheet_names
all_sheets.remove("Sheet1")

In [ ]:
# Define shortlist and tier mapping
short_list = [
    "Chase McQueen", "Morgan Pearson", "John Reed", 
    "Reese Vannerson", "Sullivan Middaugh",
    "Taylor Spivey", "Gwen Jorgensen", "Erika Ackerlund", "Keller Norland"
]

def load_and_clean(df: pd.DataFrame, sheet_name: str, tier: float) -> pd.DataFrame:
    # Normalize column names
    df.columns = df.columns.str.replace('\n', ' ').str.strip()
    # Identify & rename the athlete name column
    name_cols = [c for c in df.columns if "name" in c.lower()]
    if not name_cols:
        raise ValueError(f"No athlete column found in '{sheet_name}'")
    df = df.rename(columns={name_cols[0]: "Athlete"})
    # Add metadata
    df["Event"] = sheet_name
    df["Tier"] = tier
    # Filter to shortlist
    df = df[df["Athlete"].isin(short_list)]
    return df



In [33]:
# Lists to collect cleaned data
cleaned_individual = []
cleaned_relaylegs = []

for sheet_name in all_sheets:
    # Determine tier weight
    tier = 1.0 if "WTCS" in sheet_name else 0.6

    if sheet_name == "WTCS Abu Dhabi":
        # Split into two tables by header rows (0 and 7)
        # Table 1: Individual Race (rows 0-6)
        df1 = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            header=1,
            nrows=6
        )
        # Table 2: Mixed Relay "similar leg" (rows 7+)
        df2 = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            header=8
        )
        # Clean both
        cleaned_individual.append(load_and_clean(df1, sheet_name, tier))
        cleaned_relaylegs.append(load_and_clean(df2, sheet_name + "_RelayLeg", tier))
    else:
        # Regular single table sheets
        df = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            header=0
        )
        cleaned_individual.append(load_and_clean(df, sheet_name, tier))

In [35]:
cleaned_individual

[           Athlete Swim Place Swim Time Swim Split from Swim Lead T1 Place  \
 0    Taylor Spivey        3rd  09:14:00                      +:18       14   
 1  Erika Ackerlund       27th  09:30:00                      +:34       39   
 2   Gwen Jorgensen       18th  09:25:00                      +:29       36   
 3   Morgan Pearson       19th  08:21:00                      +:22      5th   
 4        John Reed       23rd  08:24:00                      +:25     25th   
 
   T1 Time T1 Split from Lead   Bike Analysis T2 Place T2 Time  \
 0     :40               +:03 1 days 04:13:00     24th     :25   
 1     :44               +:07 1 days 03:50:00     23rd     :24   
 2     :43               +:06 1 days 05:16:00     22nd     :24   
 3     :35               +:01 1 days 01:39:00     16th     :22   
 4     :38               +:03 1 days 01:36:00     39th     :24   
 
   T2 Split from Lead Run Place Run Split Run split from run leader  \
 0               +:05      14th  16:49:00              

In [ ]:
from datetime import datetime

# List of sheets with known time format issues
fix_time_sheets = [
    "WC Chengdu", "WTCS Yokohama", "WC Samarkaland", "WTCS Alghero", "Huatulco WC"
]

def fix_run_time(val):
    # Only fix if value is a string and matches the pattern
    if isinstance(val, str) and val.count(":") == 2:
        h, m, s = val.split(":")
        if int(h) > 10:  # Unlikely to be a real hour value for a run
            return f"00:{h.zfill(2)}:{m.zfill(2)}"
    return val

for sheet_name, df in sheets.items():
    # Assign tier weight
    tier = 1.0 if "WTCS" in sheet_name else 0.6
    
    # Normalize column names
    df.columns = df.columns.str.replace('\n', ' ').str.strip()
    
    # Identify & rename the athlete name column
    name_cols = [c for c in df.columns if "name" in c.lower()]
    if not name_cols:
        raise ValueError(f"No athlete column found in '{sheet_name}'")
    df = df.rename(columns={name_cols[0]: "Athlete"})
    
    # Fix run time format for specific sheets
    if sheet_name in fix_time_sheets:
        for col in df.columns:
            if "run" in col.lower():
                df[col] = df[col].apply(fix_run_time)
    
    # Print the DataFrame before filtering
    #print(f"Sheet: {sheet_name} - Athletes before filtering:")
    print(df["Athlete"].tolist())
    #print(df)
    
    # Add metadata columns
    df["Event"] = sheet_name
    df["Tier"] = tier
    
    # Filter to shortlist
    df = df[df["Athlete"].isin(short_list)]
    
    cleaned.append(df)




['Swim', '3rd', '27th', '18th', '19th', '23rd', nan, 'Swim Position Per Leg', '5th', '9th', '5th', '3rd']
['Reese Vannerson', 'Sullivan Middaugh', 'John Reed', 'Keller Norland', 'Erika Ackerlund', 'Danielle Orie']
['Reese Vannerson', 'Keller Norland']
['Chase McQueen', 'John Reed', 'Darr Smith', 'Morgan Pearson', 'Gwen Jorgensen', 'Taylor Spivey', 'Gina Sereno']
['Keller Norland', 'Reese Vannerson', 'Braxton Legg', 'Danielle Orie']
['Chase McQueen', 'John Reed', 'Seth Rider', 'Darr Smith', 'Summer Rappaport', 'Gwen Jorgensen']
['Erika Ackerlund', 'Gina Sereno', 'Naomi Ruff', 'Tamara Gorman']
